# LLM Zoomcamp 2025 — Homework 1 (Introduction)

**Elasticsearch:** From this folder run `docker compose up -d` (image **8.17.6**, data under repo `docker-data/volumes/elasticsearch-hw-data`).  
Do not run together with the course stack in `LLM/01/2025/docker-compose.elasticsearch.yml` — port **9200** conflict.

**Lecture notebook:** `../module-1-practice.ipynb` (parent `Introduction` folder).

Links: see `README.md` in this folder.

## Q1. Running Elastic

Cluster info: `version.build_hash`. Example: `curl http://127.0.0.1:9200`

In [ ]:
import requests

info = requests.get("http://127.0.0.1:9200", timeout=60).json()
info["version"]["number"], info["version"]["build_hash"]

## Getting the data

`pip install requests`

In [ ]:
import requests

# 2025 cohort path — `main/01-intro/documents.json` returns 404 (body "404: Not Found"),
# which breaks json() with "Extra data" because `404` parses as a number then `:` is invalid.
docs_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/01-intro/documents.json"

# Send a GET request to fetch the document data with a 60-second timeout
docs_response = requests.get(docs_url, timeout=60)

# Ensure the request was successful; raises an exception for 4XX or 5XX status codes
docs_response.raise_for_status()

# Parse the JSON response into a Python list of dictionaries
documents_raw = docs_response.json()

# Initialize an empty list to hold the flattened and processed documents
documents = []

# Iterate through each course entry in the raw data
for course in documents_raw:
    course_name = course["course"] # Extract the course ID (e.g., 'llm-zoomcamp')
    
    # Iterate through each specific FAQ entry within the current course
    for doc in course["documents"]:
        doc["course"] = course_name # Enrich each document with its parent course name
        documents.append(doc)       # Store the flattened document in the final list

# Final check: The length of the documents list should be exactly 948
len(documents)

## Q2. Indexing

`course` → `keyword`, other fields → `text`. `pip install elasticsearch`  
Which client method adds documents? (quiz choices: insert / index / put / add)

In [ ]:
from elasticsearch import Elasticsearch

# Initialize the Elasticsearch client
# "http://127.0.0.1:9200" is the address of the Docker container mapped to your localhost.
# request_timeout=120 ensures the client waits up to 120 seconds for a response, 
# which is helpful if the ElasticSearch container is still warming up.
es_client = Elasticsearch(
    "http://127.0.0.1:9200",
    request_timeout=120,
    retry_on_timeout=True,
    max_retries=3,
)

# Retrieve and print information about the cluster
# This confirms that the connection is successful and shows the version/cluster name.
es_client.info()

In [ ]:
# Configuration for the Elasticsearch index
index_settings = {
    "settings": {
        "number_of_shards": 1, 
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},     # Full-text search field
            "section": {"type": "text"},  # Full-text search field
            "question": {"type": "text"}, # Full-text search field
            "course": {"type": "keyword"},# Exact match field (Required for Q2)
        }
    },
}

index_name = "course-questions"

import time
from elasticsearch.exceptions import ApiError

# info() can succeed while the cluster still returns 503 / master_not_discovered on index APIs.
# Wait until health is at least yellow before delete/create.
print("Waiting for cluster (yellow/green) before index operations...")
for i in range(12):
    try:
        health = es_client.cluster.health(
            wait_for_status="yellow",
            timeout="30s",
            request_timeout=60,
        )
        print(f"Cluster ready: status={health['status']}")
        break
    except Exception as exc:
        print(f"Not ready yet ({i + 1}/12): {type(exc).__name__}: {exc}")
        time.sleep(3)
else:
    raise RuntimeError(
        "Cluster did not reach yellow in time. Try: docker compose restart in llm_01_hw, wait, re-run this cell."
    )

# Avoid indices.exists (HEAD) during boot — use delete and ignore 404.
try:
    es_client.indices.delete(index=index_name)
    print(f"Deleted existing index '{index_name}'.")
except ApiError as e:
    if e.meta.status != 404:
        raise

es_client.indices.create(index=index_name, body=index_settings)
print(f"Created index '{index_name}'.")

In [ ]:
from tqdm.auto import tqdm

# Iterate through the list of processed documents with a progress bar
# tqdm provides a visual feedback of the indexing process
for doc in tqdm(documents):
    # Use the 'index' function to add each document to the Elasticsearch index
    # This function is the correct answer for Q2
    es_client.index(index=index_name, document=doc)

## Q3. Searching

Query: `"How do execute a command on a Kubernetes pod?"`  
Fields: `question`, `text` only; boost `question` by **4**; `"type": "best_fields"`.  
Report top hit `_score` (compare to quiz options).

In [ ]:
query = "How do execute a command on a Kubernetes pod?"

# Construct the search query with multi_match
search_query = {
    "size": 5, # Show top 5 results
    "query": {
        # 'bool' combines multiple query clauses (like AND/OR in SQL)
        "bool": {
            # 'must' means the document MUST match this clause to be included
            # It also contributes to the relevance score (_score)
            "must": {
                # 'multi_match' allows searching for a term across multiple fields
                "multi_match": {
                    "query": query,
                    # question field is boosted by 4 as required by Q3
                    "fields": ["question^4", "text"],
                    # best_fields takes the score from the single best matching field
                    "type": "best_fields"
                }
            }
        }
    }
}

# Execute the search in the Elasticsearch index
search_results = es_client.search(index=index_name, body=search_query)
print(search_results)

# Retrieve the score of the top ranking result
# Look at the _score field to answer Q3
score = search_results["hits"]["hits"][0]["_score"]
print(score)

## Q4. Filtering

Query: `"How do copy a file to a Docker container?"`  
Filter: `machine-learning-zoomcamp` only. `size`: **3**.  
Third hit: which **question** string? (multiple choice in the assignment)

In [ ]:
query = "How do copy a file to a Docker container?"

# Define search query
search_query = {
    "size": 3,  # Retrieve only top 3 results
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"], # Boost question field by factor of 4
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp" # Exact match filtering for the course
                }
            }
        }
    }
}

search_results = es_client.search(index=index_name, body=search_query)
[h["_source"]["question"] for h in search_results["hits"]["hits"]]

## Q5. Building a prompt

Use **Q4** hits with `context_template` and `prompt_template`. `len(prompt)`

In [ ]:
# 1. Define templates and initial variables
context_template = """
Q: {question}
A: {text}
""".strip()

prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

query = "How do copy a file to a Docker container?"

# 2. [FILL TODO] Build context from search_results
context_entries = []

# Iterate through the top 3 search hits from Q4
for hit in search_results['hits']['hits']:
    doc = hit['_source']
    # Format each record using the context_template
    entry = context_template.format(question=doc['question'], text=doc['text'])
    context_entries.append(entry)

# Separate context items with two line breaks (\n\n)
context = "\n\n".join(context_entries)

# 3. Build the final prompt and calculate length
prompt = prompt_template.format(question=query, context=context)

# Print the final character count
print(len(prompt))

## Q6. Tokens

`pip install tiktoken`  
`encoding = tiktoken.encoding_for_model("gpt-4o")` then `len(encoding.encode(prompt))`

In [ ]:
!pip install tiktoken

In [ ]:
import subprocess
import sys

# Use this kernel's interpreter (safer than `!pip` when PATH/kernels differ).
subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken"])

import tiktoken
print("tiktoken ready — run the next cell.")

In [ ]:
import tiktoken

# 1. Initialize the encoding for the specific model (gpt-4o)
encoding = tiktoken.encoding_for_model("gpt-4o")

# 2. Encode the prompt into tokens
tokens = encoding.encode(prompt)

# 3. Calculate and print the number of tokens
num_tokens = len(tokens)
print(f"The number of tokens in the prompt: {num_tokens}")

# Optional: To see how a specific token looks as a word (e.g., token 63842)
# word = encoding.decode_single_token_bytes(63842)
# print(f"Token 63842 corresponds to: {word}")

## Bonus 1 (ungraded): Live API

Send `prompt` to OpenAI (or swap in Ollama). Use the same **`gpt-4o`** family as token counting in Q6; cost worksheet is **Bonus 2**.  
Put your key in **`LLM/.env`** as `OPENAI_API_KEY=...` or export it — the code cell reads `.env` for this Jupyter kernel.

In [ ]:
import subprocess
import sys

# Prefer --no-cache-dir on slow disks to avoid long pip cache writes.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "openai",
        "python-dotenv",
    ]
)

print("Installed openai + python-dotenv (this kernel).")

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

# Load keys into *this* process (Anaconda Jupyter often lacks OS-wide env).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "python-dotenv"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

from dotenv import load_dotenv
from openai import OpenAI

_loaded = False
_cwd = Path.cwd().resolve()
for d in [_cwd, *_cwd.parents]:
    for cand in (d / ".env", d / "LLM" / ".env"):
        if cand.is_file():
            load_dotenv(cand)
            _loaded = True
            break
    if _loaded:
        break
if not _loaded:
    load_dotenv()

_api_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
if not _api_key:
    _api_key = getpass("OPENAI_API_KEY (hidden): ").strip()
if not _api_key:
    raise RuntimeError("Missing OPENAI_API_KEY — create LLM/.env or paste above.")

client = OpenAI(api_key=_api_key)

# Sends the prompt built in Q5 to the same model family counted in Q6.
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
)

print(response.choices[0].message.content)
if getattr(response, "usage", None) is not None:
    print(
        "usage:",
        response.usage.prompt_tokens,
        "prompt +",
        response.usage.completion_tokens,
        "completion tokens",
    )

In [ ]:
import json

# Optional offline path — main notebook pulls JSON from GitHub when available.
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

# Flatten nested course payloads into `documents`
documents = []
for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [ ]:
from elasticsearch import Elasticsearch

# Homework Elasticsearch from docker-compose (published on localhost:9200).
es_client = Elasticsearch('http://localhost:9200') 

In [ ]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

# Idempotent teardown + create for repeatable notebook runs.
es_client.indices.delete(index=index_name, ignore_unavailable=True)
es_client.indices.create(index=index_name, body=index_settings)


In [ ]:
from tqdm.auto import tqdm

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

## Bonus 2 (ungraded): Calculating the costs

Suppose that **on average** per request we send **150 tokens** (input) and receive **250 tokens** (output).

**How much will it cost to run 1000 requests?**

Example **gpt-4o** rates from the homework materials (**June 17**):

- **Input:** $0.005 / 1K tokens
- **Output:** $0.015 / 1K tokens

**Redo** with **`num_tokens` from Q6** and **`response.usage`** from Bonus 1 (or estimate completion tokens with tiktoken on the assistant text), or keep the fixed 150 / 250 averages.

Some quizzes label the cost follow-up as **Q7** (with Q6 token counts).

In [ ]:
# File Name: calculate_api_cost_advanced.py
# Description: Advanced GPT-4o API Cost Estimator with Functions and Error Handling

# ---------------------------------------------------------
# SECTION 1: Fixed Configuration (Default Values)
# ---------------------------------------------------------
# Based on the homework scenario: 1,000 requests with fixed averages
requests_count = 1000
tokens_in_per_request = 150
tokens_out_per_request = 250

# Pricing for GPT-4o (Rates as of June 17, 2026)
usd_per_1k_input = 0.005
usd_per_1k_output = 0.015


# ---------------------------------------------------------
# SECTION 2: Reusable Calculation Function
# ---------------------------------------------------------
def dollars_for_tokens(n_prompt_tokens: int, n_completion_tokens: int, n_requests: int = 1) -> float:
    """
    Calculates the total cost based on token counts and number of requests.
    Formula: Requests * ((Input/1000 * Rate) + (Output/1000 * Rate))
    """
    input_cost = (n_prompt_tokens / 1000) * usd_per_1k_input
    output_cost = (n_completion_tokens / 1000) * usd_per_1k_output
    return n_requests * (input_cost + output_cost)


# ---------------------------------------------------------
# SECTION 3: Standard Output (Fixed Average Results)
# ---------------------------------------------------------
fixed_total = dollars_for_tokens(tokens_in_per_request, tokens_out_per_request, requests_count)

print("=" * 60)
print(f"COST ESTIMATE (Based on Fixed Averages)")
print("-" * 60)
print(f"Total Requests:      {requests_count}")
print(f"Average In/Out:      {tokens_in_per_request} / {tokens_out_per_request} tokens")
print(f"Estimated Total:     ${fixed_total:.2f} USD")
print("=" * 60)


# ---------------------------------------------------------
# SECTION 4: Dynamic Check (For Q6/Q7 Variable Integration)
# ---------------------------------------------------------
# This part attempts to use actual token data if you've run previous cells.
try:
    # Check if 'num_tokens' from Q6 exists
    n_in_q6 = int(num_tokens)
    
    # Try to extract actual usage from the API 'response' object
    u = getattr(response, "usage", None)

    if u is not None and hasattr(u, "prompt_tokens"):
        pi, co = u.prompt_tokens, u.completion_tokens
        one_req = dollars_for_tokens(pi, co, 1)
        thousand_req = dollars_for_tokens(pi, co, requests_count)
        
        print("\n[BONUS] Actual API Usage Detection:")
        print(f" - Per Request: {pi} (In) + {co} (Out)")
        print(f" - Cost for 1:  ${one_req:.6f}")
        print(f" - Cost for 1k: ${thousand_req:.2f}")

except NameError:
    # Silently skip if variables from previous cells are not found
    pass

print("\n(Note: Calculated using GPT-4o rates from June 17)")
print("=" * 60)